# 第四课：迁移学习（预训练模型 + 数据增强可视化）

**学习目标**：用 ImageNet 预训练的 MobileNetV2，通过迁移学习快速适配新任务。这是你钢坯检测项目的标配。

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import os; os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

## 1. 加载预训练模型（MobileNetV2）

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(96,96,3), include_top=False, weights='imagenet')
base_model.trainable = False
print('层数:', len(base_model.layers))
print('参数量: {:,}'.format(base_model.count_params()))
print('特征输出shape:', base_model.output_shape)

## 2. 数据增强可视化

数据少的时候，数据增强是防止过拟合的关键。

In [ ]:
# 造一张模拟图片
np.random.seed(42)
sample_img = np.random.rand(96, 96, 3).astype(np.float32)
# 加一些结构（模拟缺陷）
sample_img[30:50, 30:50] = 0.9
sample_img[60:70, 20:80] = 0.1

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.3),
])

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes[0,0].imshow(sample_img); axes[0,0].set_title('原图'); axes[0,0].axis('off')
for i, ax in enumerate(axes.flat[1:]):
    augmented = data_augmentation(sample_img[np.newaxis], training=True)[0].numpy()
    ax.imshow(np.clip(augmented, 0, 1))
    ax.set_title('增强%d'%(i+1)); ax.axis('off')
plt.suptitle('数据增强效果（同一张图的不同变换）', fontsize=14)
plt.tight_layout(); plt.show()

## 3. 构建迁移学习模型

In [ ]:
inputs = tf.keras.Input(shape=(96,96,3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

total = model.count_params()
trainable = sum(w.shape.num_elements() for w in model.trainable_weights)
print('总参数: {:,}'.format(total))
print('可训练: {:,} ({:.2f}%)'.format(trainable, trainable/total*100))
print('冻结: {:,} ({:.2f}%)'.format(total-trainable, (total-trainable)/total*100))

## 4. 模拟数据训练（用随机数据模拟你的钢坯数据集）

In [ ]:
np.random.seed(42)
X_tr = np.random.rand(500, 96, 96, 3).astype(np.float32)
y_tr = np.random.randint(0, 10, 500)
X_val = np.random.rand(100, 96, 96, 3).astype(np.float32)
y_val = np.random.randint(0, 10, 100)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist1 = model.fit(X_tr, y_tr, validation_data=(X_val,y_val),
                  epochs=5, batch_size=32, verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].plot(hist1.history['loss'], 'b-o', ms=4, label='训练')
axes[0].plot(hist1.history['val_loss'], 'r-o', ms=4, label='验证')
axes[0].set_title('第一阶段：只训练分类头 (lr=1e-3)')
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_xlabel('Epoch')
axes[1].plot(hist1.history['accuracy'], 'b-o', ms=4, label='训练')
axes[1].plot(hist1.history['val_accuracy'], 'r-o', ms=4, label='验证')
axes[1].set_title('准确率')
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()

## 5. 微调（解冻部分层，用更小学习率）

In [ ]:
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 20
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist2 = model.fit(X_tr, y_tr, validation_data=(X_val,y_val),
                  epochs=3, batch_size=32, verbose=0)

# 合并两阶段曲线
all_loss = hist1.history['loss'] + hist2.history['loss']
all_val_loss = hist1.history['val_loss'] + hist2.history['val_loss']
all_acc = hist1.history['accuracy'] + hist2.history['accuracy']
all_val_acc = hist1.history['val_accuracy'] + hist2.history['val_accuracy']

fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].plot(all_loss, 'b-o', ms=4, label='训练损失')
axes[0].plot(all_val_loss, 'r-o', ms=4, label='验证损失')
axes[0].axvline(x=len(hist1.history['loss'])-0.5, color='g', ls='--', label='开始微调')
axes[0].set_title('两阶段训练损失曲线')
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_xlabel('Epoch')
axes[1].plot(all_acc, 'b-o', ms=4, label='训练准确率')
axes[1].plot(all_val_acc, 'r-o', ms=4, label='验证准确率')
axes[1].axvline(x=len(hist1.history['accuracy'])-0.5, color='g', ls='--', label='开始微调')
axes[1].set_title('两阶段准确率曲线')
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()

## 6. 预训练模型特征提取可视化

把一张图通过预训练模型，看看提取的特征是什么样的。

In [ ]:
# 用模拟图片
demo_img = np.random.rand(1, 96, 96, 3).astype(np.float32)
demo_img[0, 30:50, 30:50] = 0.9

preprocessed = tf.keras.applications.mobilenet_v2.preprocess_input(demo_img)
features = base_model(preprocessed, training=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(demo_img[0]); axes[0].set_title('输入图片'); axes[0].axis('off')
# 展示特征图的前16个通道（3x3空间）
feat_vis = np.zeros((3*4, 3*4))
for i in range(16):
    r, c = i//4, i%4
    feat_vis[r*3:(r+1)*3, c*3:(c+1)*3] = features[0,:,:,i]
axes[1].imshow(feat_vis, cmap='viridis'); axes[1].set_title('MobileNetV2提取的特征(前16通道)'); axes[1].axis('off')
plt.tight_layout(); plt.show()
print('特征图shape:', features.shape)

## 7. 可用预训练模型对比

In [ ]:
models_info = [
    ('MobileNetV2', 3.5, '轻量，移动端/边缘设备首选'),
    ('EfficientNetB0', 5.3, '效率高，参数少'),
    ('ResNet50', 25.6, '经典残差网络'),
    ('ResNet50V2', 25.6, 'ResNet改进版'),
    ('DenseNet121', 8.1, '密集连接'),
    ('InceptionV3', 23.9, 'GoogleNet变体'),
    ('VGG16', 138, '经典但笨重'),
]
names = [m[0] for m in models_info]
params = [m[1] for m in models_info]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names, params, color='steelblue')
ax.set_xlabel('参数量 (M)'); ax.set_title('Keras内置预训练模型参数量对比')
for bar, (_, _, desc) in zip(bars, models_info):
    ax.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2, desc, va='center', fontsize=9)
ax.set_xlim(0, 160); ax.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

## 第四课小结

**迁移学习两阶段**：
1. **特征提取**：冻结 backbone，只训练分类头（lr=1e-3）
2. **微调**：解冻顶部几层，用小学习率（lr=1e-5）精调

**对你的钢坯检测项目**：
- 数据少 → 必须用迁移学习 + 数据增强
- 边缘部署 → 选 MobileNetV2 / EfficientNetB0
- 训练完导出 SavedModel → TFLite → 量化部署

全部课程完成！你已经掌握了 TF 的核心用法，可以开始你的项目了。